In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

In [ ]:
# reset defalult plotting values
plt.rcParams['figure.figsize'] = (10, 5)
plt.rc('font', family='sans-serif')
plt.rc('axes', labelsize=14)
plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=16)
plt.rc('axes', titleweight='bold')

# ASTR 350: Astronomical Techniques
# Lecture 27


### Prof. Robert Quimby <br> April 28, 2026

&copy; 2026 Robert Quimby

## Review Quiz #21

## Think about all the uncertainty in your measurements

<img width="500" src="media/percision_vs_accuracy.png">

## Example: RR Lyrae Period-Luminosity Relation

In Project 4 you measured how the periods of RR Lyrae correlate with luminosity
- what contributes to the statistical uncertainty in the measurement?
- what may affect the accuracy of the measurement?

## Effects of sampling on statistical uncertainties

## Example 1: focusing a telescope

In [ ]:
def focus_model(focus, *params):
    best_focus, minfwhm, a = params
    return minfwhm + a * (focus - best_focus) ** 2

In [ ]:
params = 13000, 2.3, 1.5e-4
focus = np.linspace(params[0] - 300, params[0] + 300, 100)
fwhm = focus_model(focus, *params)
plt.plot(focus, fwhm, c='k');

## What is the best way to sample the focus?

- it takes time to record each focus image
- so we want to take the fewest images necessary to precisely measure the focus
- if you think the focus is at 13000 and you only have time to take 3 images, what focus values should you choose?

In [ ]:
# generate some fake data
test_focus = np.array([12800, 13000, 13200])
test_fwhm = ????
plt.plot(focus, fwhm, c='k')
plt.scatter(test_focus, test_fwhm);

In [ ]:
# find the best-fit parameters using chi-square minimization
from scipy.optimize import fmin
def get_residuals(params, x, y):
    return np.sum((y - ????)**2)

initial_guess = params
best_params = fmin(get_residuals, initial_guess, args=(test_focus, test_fwhm), disp=False)
best_params[0]

In [ ]:
plt.plot(focus, fwhm, c='k', label='true relation')
plt.plot(focus, focus_model(focus, *best_params), label='best-fit relation');
plt.axvline(params[0], ls='dashed', c='k')
plt.axvline(best_params[0], ls='dashed', c='C0')
plt.legend();

In [ ]:
ntries = ????
focuses = np.zeros(ntries)
for i in range(ntries):
    test_fwhm = focus_model(test_focus, *params) + np.random.normal(0, 0.5, size=test_focus.size)
    best_params = fmin(get_residuals, initial_guess, args=(test_focus, test_fwhm), disp=False)
    focuses[i] = ????

In [ ]:
plt.hist(focuses, bins=25);
focuses.std()

## Example 2: Sampling a (slightly) more complex relation

In [ ]:
# try more complex model (linear + sine)
def true_func(x):
    return 3.4 * x + 4 * np.sin(x) + 25

In [ ]:
# plot the true relation
testx = np.linspace(-9, 9, 100)
testy = true_func(testx)
plt.plot(testx, testy, c='k', lw=3);

In [ ]:
# generate some fake data
xs = np.array([-5, 2.3, 7.5])
ey = 1
ys = true_func(xs) + np.random.normal(0, ey, size=xs.size)

In [ ]:
# least squares fit to the data
X = np.matrix([[x, np.sin(x), 1] for x in xs])
Y = np.matrix([[y] for y in ys])
p = (X.T * X).I * X.T * Y
p

In [ ]:
# plot the true relation
plt.plot(testx, testy, c='k', lw=3);

# mark the data
plt.errorbar(xs, ys, yerr=ey, ls='none', marker='o');

# plot the derived relation
X = np.matrix([[x, np.sin(x), 1] for x in testx])
modelY = X * p
plt.plot(testx, modelY);

## What sampling of data minimizes the amplitude uncertainty?

In [ ]:
def get_best_fit(func, xs, ys, ey=1):
    X = np.matrix([[x, np.sin(x), 1] for x in xs])
    Y = np.matrix([[y] for y in ys])
    p = (X.T * X).I * X.T * Y
    return p

In [ ]:
ntries = ????
amplitudes = np.zeros(ntries)
params = []
for i in range(ntries):
    ys = true_func(xs) + np.random.normal(0, ey, size=xs.size)
    p = get_best_fit(true_func, xs, ys)
    params.append(p)
    amplitudes[i] = p.A1[1]

In [ ]:
plt.hist(amplitudes, bins=25);
amplitudes.std()

In [ ]:
# plot the derived relations
for p in random.sample(params, ????):
    X = np.matrix([[x, np.sin(x), 1] for x in testx])
    modelY = X * p
    plt.plot(testx, modelY, c='C1', alpha=0.1);
    
# plot the true relation
plt.plot(testx, testy, c='k', lw=3)

# mark the data
plt.errorbar(xs, ys, yerr=ey, ls='none', marker='o');

## Questions on the final project?